# DeepCare 07: Export the model for the demo app

The Streamlit app does not use TensorFlow. It runs the trained MLP as plain NumPy, which keeps deployment light and makes per-person SHAP explanations fast.

This notebook:
1. Exports the MLP weights, the scaler parameters and the Platt calibrator into small files (`mlp_weights.npz`, `app_bundle.json`).
2. Verifies **end to end** that the NumPy pipeline (raw features, BMI clipping, scaling, MLP, calibration) reproduces the Keras model's calibrated test probabilities. The verification uses the exact `inference.py` module the app uses.
3. Zips the files the app needs.

Files the app needs in `models/`: `mlp_weights.npz`, `app_bundle.json`, `app_config.json` and `shap_background.npz` (the last two come from notebook 06).

## 1. Load the trained pieces

In [ ]:
import os, json, joblib, shutil, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/deepcare'
D, R, M = f'{PROJECT_DIR}/data', f'{PROJECT_DIR}/results', f'{PROJECT_DIR}/models'
TARGET = 'Diabetes_binary'

model = tf.keras.models.load_model(f'{M}/mlp.keras')
scaler = joblib.load(f'{PROJECT_DIR}/scaler.joblib')
platt = joblib.load(f'{R}/calibrators.joblib')['mlp']['platt']
prep = json.load(open(f'{PROJECT_DIR}/prep_config.json'))
for f in ['app_config.json', 'shap_background.npz']:
    assert os.path.exists(f'{M}/{f}'), f'missing {f}: run notebook 06 (and the background-saving cell) first'
model.summary()

## 2. Export weights, scaler and calibrator

Every layer type in the trained network (Dense, BatchNormalization, ReLU, Dropout) is converted; dropout does nothing at inference time. The export raises an error on any layer it does not know, so a silent mismatch is not possible.

In [ ]:
spec, arrays = [], {}
for i, layer in enumerate(model.layers):
    cls = layer.__class__.__name__
    if cls == 'Dense':
        kernel, bias = layer.get_weights()
        arrays[f'w{i}'], arrays[f'b{i}'] = kernel, bias
        spec.append({'type': 'dense', 'w': f'w{i}', 'b': f'b{i}', 'activation': layer.activation.__name__})
    elif cls == 'BatchNormalization':
        gamma, beta, mean, var = layer.get_weights()          # gamma, beta, moving mean, moving variance
        arrays.update({f'g{i}': gamma, f'be{i}': beta, f'mm{i}': mean, f'mv{i}': var})
        spec.append({'type': 'bn', 'g': f'g{i}', 'be': f'be{i}', 'mm': f'mm{i}', 'mv': f'mv{i}',
                     'eps': float(layer.epsilon)})
    elif cls == 'ReLU':
        spec.append({'type': 'relu'})
    elif cls in ('Dropout', 'InputLayer'):
        continue
    else:
        raise ValueError(f'unsupported layer: {cls}')

NUM = prep['NUM']
bundle = {
    'layers': spec,
    'scaler': {'features': NUM, 'mean': [float(v) for v in scaler.mean_], 'scale': [float(v) for v in scaler.scale_]},
    'platt': {'coef': float(platt.coef_[0, 0]), 'intercept': float(platt.intercept_[0])},
}
np.savez(f'{M}/mlp_weights.npz', **arrays)
json.dump(bundle, open(f'{M}/app_bundle.json', 'w'), indent=2)
print('layers exported:', [s['type'] for s in spec])
print('platt:', bundle['platt'])

## 3. Verify the NumPy pipeline against the saved Keras results

`inference.py` is written out below exactly as it will ship with the app, then run on the **raw** test rows. The result must match the calibrated probabilities from notebook 05 (Platt scaling of the saved MLP test probabilities).

In [ ]:
INFERENCE_SRC = r'''"""Pure-numpy inference for the DeepCare Keras MLP (no TensorFlow or scikit-learn needed).

Pipeline for one or more people given as RAW feature values in cfg['feature_order']:
    raw -> clip BMI -> standardize numeric columns -> MLP forward pass -> Platt calibration
"""
import json
from pathlib import Path

import numpy as np


def load_bundle(models_dir):
    d = Path(models_dir)
    bundle = json.loads((d / 'app_bundle.json').read_text())
    weights = {k: v.astype('float64') for k, v in np.load(d / 'mlp_weights.npz').items()}
    cfg = json.loads((d / 'app_config.json').read_text())
    background = np.load(d / 'shap_background.npz')['data'].astype('float64')   # already in scaled model space
    return {'bundle': bundle, 'weights': weights, 'cfg': cfg, 'background': background}


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def preprocess(raw, art):
    """RAW values, shape (n, n_features) in feature_order -> scaled model input."""
    cfg, sc = art['cfg'], art['bundle']['scaler']
    order = cfg['feature_order']
    x = np.array(raw, dtype='float64', copy=True)
    if x.ndim == 1:
        x = x[None, :]
    j = order.index('BMI')
    lo, hi = cfg['bmi_clip']
    x[:, j] = np.clip(x[:, j], lo, hi)
    for name, mean, scale in zip(sc['features'], sc['mean'], sc['scale']):
        k = order.index(name)
        x[:, k] = (x[:, k] - mean) / scale
    return x


def raw_score(x_scaled, art):
    """MLP forward pass: probability before calibration."""
    W = art['weights']
    h = np.asarray(x_scaled, dtype='float64')
    for layer in art['bundle']['layers']:
        t = layer['type']
        if t == 'dense':
            h = h @ W[layer['w']] + W[layer['b']]
            if layer['activation'] == 'sigmoid':
                h = _sigmoid(h)
            elif layer['activation'] == 'relu':
                h = np.maximum(h, 0.0)
            elif layer['activation'] != 'linear':
                raise ValueError(f"unsupported activation {layer['activation']}")
        elif t == 'bn':
            h = (h - W[layer['mm']]) / np.sqrt(W[layer['mv']] + layer['eps']) * W[layer['g']] + W[layer['be']]
        elif t == 'relu':
            h = np.maximum(h, 0.0)
        else:
            raise ValueError(f'unsupported layer type {t}')
    return h.ravel()


def calibrate(p, art):
    """Platt scaling fitted on the validation set: sigmoid(coef * logit(p) + intercept)."""
    p = np.clip(np.asarray(p, dtype='float64'), 1e-6, 1 - 1e-6)
    pl = art['bundle']['platt']
    return _sigmoid(pl['coef'] * np.log(p / (1 - p)) + pl['intercept'])


def predict_risk(x_scaled, art):
    return calibrate(raw_score(x_scaled, art), art)
'''

os.makedirs('/content/app_check', exist_ok=True)
open('/content/app_check/inference.py', 'w').write(INFERENCE_SRC)
sys.path.insert(0, '/content/app_check')
import importlib, inference
importlib.reload(inference)

In [ ]:
art = inference.load_bundle(M)

te = pd.read_csv(f'{D}/test_raw.csv')
Xte_raw, yte = te.drop(columns=[TARGET]), te[TARGET].to_numpy()
assert list(Xte_raw.columns) == art['cfg']['feature_order'], 'feature order mismatch'

x = inference.preprocess(Xte_raw.to_numpy(), art)
p_np_raw = inference.raw_score(x, art)
p_np_cal = inference.predict_risk(x, art)

z = np.load(f'{R}/probs_mlp.npz')
logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))).reshape(-1, 1)
p_ref_cal = platt.predict_proba(logit(z['test']))[:, 1]

print('max |numpy - saved Keras| raw probability       :', float(np.abs(p_np_raw - z['test']).max()))
print('max |numpy - saved Keras| calibrated probability:', float(np.abs(p_np_cal - p_ref_cal).max()))
print('mean calibrated risk:', round(float(p_np_cal.mean()), 4), '| observed rate:', round(float(yte.mean()), 4))
assert np.abs(p_np_cal - p_ref_cal).max() < 1e-4, 'NumPy pipeline does not reproduce the Keras results'
print('OK: the app pipeline reproduces the evaluated model')

## 4. Zip the files for the repo

In [ ]:
bundle_dir = '/content/deepcare_app_files/models'
shutil.rmtree('/content/deepcare_app_files', ignore_errors=True)
os.makedirs(bundle_dir)
for f in ['mlp_weights.npz', 'app_bundle.json', 'app_config.json', 'shap_background.npz']:
    shutil.copy(f'{M}/{f}', bundle_dir)
archive = shutil.make_archive(f'{PROJECT_DIR}/deepcare_app_files', 'zip', '/content/deepcare_app_files')
print('saved', archive)
print({f: os.path.getsize(f'{bundle_dir}/{f}') for f in os.listdir(bundle_dir)})

from google.colab import files
files.download(archive)      # unzip at the repository root so the files land in models/